In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(
    "../data/raw/flood_forecasting/flood_forecasting_hourly.csv"
)

df["timestamp"] = pd.to_datetime(df["timestamp"])

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (175140, 19)
            timestamp station_id  station_name  latitude  longitude  \
0 2024-01-01 00:00:00       ST01  Kaveri_North      11.9       79.8   
1 2024-01-01 01:00:00       ST01  Kaveri_North      11.9       79.8   
2 2024-01-01 02:00:00       ST01  Kaveri_North      11.9       79.8   
3 2024-01-01 03:00:00       ST01  Kaveri_North      11.9       79.8   
4 2024-01-01 04:00:00       ST01  Kaveri_North      11.9       79.8   

   rainfall_1h_mm  rainfall_3h_mm  rainfall_6h_mm  rainfall_24h_mm  \
0        0.248125        0.248125        0.248125         0.248125   
1        0.000000        0.248125        0.248125         0.248125   
2        3.053676        3.301801        3.301801         3.301801   
3        0.515474        3.569150        3.817275         3.817275   
4        3.367780        6.936930        7.185055         7.185055   

   temperature_c  humidity_pct  soil_moisture  upstream_discharge_m3s  \
0      23.802773     58.800168       0.262518      

In [2]:
features = [
    "rainfall_1h_mm",
    "rainfall_3h_mm",
    "rainfall_6h_mm",
    "rainfall_24h_mm",
    "temperature_c",
    "humidity_pct",
    "soil_moisture",
    "upstream_discharge_m3s",
    "water_level_m",
    "elevation_m",
    "latitude",
    "longitude"
]

target = "flood_next_6h"

X = df[features]
y = df[target]

print("Features:", len(features))
print("X shape:", X.shape)
print("y shape:", y.shape)

Features: 12
X shape: (175140, 12)
y shape: (175140,)


In [3]:
split_date = pd.Timestamp("2025-07-01")

train_df = df[df["timestamp"] < split_date]
test_df = df[df["timestamp"] >= split_date]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (131280, 12)
Testing : (43860, 12)


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training scaled shape:", X_train_scaled.shape)
print("Testing scaled shape :", X_test_scaled.shape)

Training scaled shape: (131280, 12)
Testing scaled shape : (43860, 12)


In [5]:
import torch

X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

torch.Size([131280, 12])
torch.Size([131280])


In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DisasterGuardBayesianNN(nn.Module):

    def __init__(
        self,
        input_features=12,
        embedding_dim=32,
        dropout_rate=0.3
    ):
        super().__init__()

        self.input_features = input_features
        self.embedding_dim = embedding_dim

        # Convert each individual feature into an embedding
        self.feature_embedding = nn.Linear(
            1,
            embedding_dim
        )

        # Self-attention across the 12 features
        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=4,
            dropout=dropout_rate,
            batch_first=True
        )

        # Feature representation
        self.fc1 = nn.Linear(
            embedding_dim,
            64
        )

        self.dropout1 = nn.Dropout(
            dropout_rate
        )

        self.fc2 = nn.Linear(
            64,
            32
        )

        self.dropout2 = nn.Dropout(
            dropout_rate
        )

        # Flood prediction
        self.output = nn.Linear(
            32,
            1
        )

    def forward(self, x):

        # x:
        # [batch_size, 12]

        # Convert each feature into a token
        x = x.unsqueeze(-1)

        # [batch_size, 12, 1]
        x = self.feature_embedding(x)

        # [batch_size, 12, 32]

        # Self-attention across features
        attended, attention_weights = self.attention(
            x,
            x,
            x
        )

        # Aggregate the 12 feature representations
        x = attended.mean(dim=1)

        # [batch_size, 32]

        x = F.relu(
            self.fc1(x)
        )

        x = self.dropout1(x)

        x = F.relu(
            self.fc2(x)
        )

        x = self.dropout2(x)

        logits = self.output(x)

        return logits.squeeze(1)

In [21]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = DisasterGuardBayesianNN(
    input_features=12,
    embedding_dim=32,
    dropout_rate=0.3
).to(device)

print(model)

DisasterGuardBayesianNN(
  (feature_embedding): Linear(in_features=1, out_features=32, bias=True)
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
  )
  (fc1): Linear(in_features=32, out_features=64, bias=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (output): Linear(in_features=32, out_features=1, bias=True)
)


In [19]:
positive = y_train.sum()
negative = len(y_train) - positive

pos_weight = torch.tensor(
    [negative / positive],
    dtype=torch.float32
).to(device)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print("Positive weight:", pos_weight.item())

Positive weight: 10.444512367248535


In [ ]:
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 1024

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Testing batches :", len(test_loader))

Training batches: 129
Testing batches : 43


In [12]:
from copy import deepcopy

epochs = 30
patience = 5

best_val_loss = float("inf")
best_model_state = None
patience_counter = 0

train_losses = []
val_losses = []

for epoch in range(epochs):

    # =========================
    # TRAINING
    # =========================
    model.train()

    running_train_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(logits, y_batch)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)


    # =========================
    # VALIDATION
    # =========================
    model.eval()

    running_val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)

            loss = criterion(logits, y_batch)

            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(test_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)


    # =========================
    # EARLY STOPPING
    # =========================
    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        best_model_state = deepcopy(
            model.state_dict()
        )

        patience_counter = 0

    else:

        patience_counter += 1


    print(
        f"Epoch [{epoch+1:02d}/{epochs}] "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )


    if patience_counter >= patience:

        print("\nEarly stopping triggered.")

        break


# Restore best model
model.load_state_dict(best_model_state)

print("\nTraining completed.")
print("Best validation loss:", best_val_loss)

Epoch [01/30] Train Loss: 1.2021 | Val Loss: 1.4782
Epoch [02/30] Train Loss: 1.2010 | Val Loss: 1.4782
Epoch [03/30] Train Loss: 1.2018 | Val Loss: 1.4782
Epoch [04/30] Train Loss: 1.2004 | Val Loss: 1.4782
Epoch [05/30] Train Loss: 1.2012 | Val Loss: 1.4782
Epoch [06/30] Train Loss: 1.1991 | Val Loss: 1.4782

Early stopping triggered.

Training completed.
Best validation loss: 1.4781674631806307
